In [ ]:
from config_path import add_to_sys_path
add_to_sys_path()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from Energy_Levels import MoleculeLevels, Calculate_TDM_evecs, branching_ratios

In [ ]:
N_g = np.arange(0, 4)
N_e = np.arange(1, 5)

g = MoleculeLevels.initialize_state(
    molecule_name='RaF', elec_state='X', vib_state=0,
    N_list=N_g, fermion_or_boson='boson',
    M_sublevels='all', M_list=[1/2],
    I_nuclei=[0, 1/2], isotope=226, round=8,
    params=None, P_values=[1/2],
)
e = MoleculeLevels.initialize_state(
    molecule_name='RaF', elec_state='A', vib_state=0,
    N_list=N_e, fermion_or_boson='boson',
    M_sublevels='all', M_list=[1/2],
    I_nuclei=[0, 1/2], isotope=226, round=8,
    params=None, P_values=[1/2],
)
g.eigensystem(0, 1e-3)
e.eigensystem(0, 1e-3)

In [ ]:
Bz = np.linspace(1e-6, 100, 401)   # Gauss; coarser than RaF X-A's 4000 to keep first run fast
g.ZeemanMap(Bz, plot=False)
e.ZeemanMap(Bz, plot=False)

assert g.evals_B.shape[0] == len(Bz) and g.evals_B.ndim == 2
assert e.evals_B.shape[0] == len(Bz) and e.evals_B.ndim == 2
assert g.evecs_B.ndim == 3 and e.evecs_B.ndim == 3

In [ ]:
def _polarization_to_p_weights(polarization):
    """Map polarization label → dict {p: weight} for coherent superposition.

    Returns weights for p in {-1, 0, +1}. Used by compute_transitions_vs_B
    to build the per-polarization TDM from per-p TDMs.
    """
    if isinstance(polarization, dict):
        out = {-1: 0.0+0j, 0: 0.0+0j, +1: 0.0+0j}
        for k, v in polarization.items():
            kk = int(k) if isinstance(k, str) else k
            out[kk] = complex(v)
        return out
    s = polarization.lower()
    if s in ('sigma+', 'sigma_plus', 's+'):  return {-1: 0+0j, 0: 0+0j, +1: 1+0j}
    if s in ('sigma-', 'sigma_minus', 's-'): return {-1: 1+0j, 0: 0+0j, +1: 0+0j}
    if s in ('pi', 'z'):                     return {-1: 0+0j, 0: 1+0j, +1: 0+0j}
    if s == 'x':
        return {-1: 1/np.sqrt(2)+0j, 0: 0+0j, +1: -1/np.sqrt(2)+0j}
    if s == 'y':
        return {-1: 1j/np.sqrt(2),    0: 0+0j, +1: 1j/np.sqrt(2)}
    raise ValueError(f"Unknown polarization {polarization!r}")


def _qn_label(state, idx):
    """Build a short label like 'N=0,J=1/2,F=1,M=+1/2' from dominant-basis QNs."""
    qn = state.q_numbers
    parts = []
    for key in ('N', 'J', 'F', 'M'):
        if key in qn:
            v = qn[key][idx]
            if abs(v - round(v)) < 1e-6:
                parts.append(f"{key}={int(round(v))}")
            else:
                parts.append(f"{key}={int(round(2*v))}/2")
    if hasattr(state, 'parities'):
        parts.append('+' if state.parities[idx] > 0 else '-')
    return ','.join(parts)


def compute_transitions_vs_B(g, e, ground_query, excited_query,
                             polarization='sigma+',
                             q_electronic=(-1, +1),
                             parity_g=None, parity_e=None,
                             tdm_thresh=1e-6):
    """Compute transition energies, spontaneous-emission BR, and excitation
    strengths vs B for selected ground/excited level pairs.

    g, e must have ZeemanMap(Bz) already run. q_electronic is the molecule-frame
    component set fixed by the electronic transition character (default (-1,+1)
    for X-A perpendicular). polarization is the lab-frame light polarization.

    Returns dict — see plan / design spec.
    """
    Bz = g.Bz
    assert np.array_equal(Bz, e.Bz), "g and e must share the same Bz grid"
    N_B = len(Bz)

    g_idx = g.select_q(ground_query, parity=parity_g)
    e_idx = e.select_q(excited_query, parity=parity_e)
    if len(g_idx) == 0 or len(e_idx) == 0:
        raise ValueError(f"Empty selection: g_idx={g_idx}, e_idx={e_idx}")

    n_g_full = g.evecs_B.shape[1]
    n_e_full = e.evecs_B.shape[1]
    p_list = (-1, 0, +1)
    TDM_full = np.zeros((N_B, 3, n_e_full, n_g_full), dtype=complex)
    for ip, p in enumerate(p_list):
        for i in range(N_B):
            TDM_full[i, ip] = Calculate_TDM_evecs(
                p, g.evecs_B[i], g, e.evecs_B[i], e, q=list(q_electronic),
            )

    TDM_sel = TDM_full[:, :, e_idx[:, None], g_idx[None, :]]   # (N_B, 3, n_e_sel, n_g_sel)

    # Spontaneous-emission BR: sum |TDM_p|^2 over photon polarization p
    num   = (np.abs(TDM_sel)**2).sum(axis=1)                       # (N_B, n_e_sel, n_g_sel)
    denom = (np.abs(TDM_full[:, :, e_idx, :])**2).sum(axis=(1, 3)) # (N_B, n_e_sel)
    BR = num / np.where(denom[:, :, None] > 0, denom[:, :, None], 1.0)
    leakage = 1.0 - BR.sum(axis=2)

    weights = _polarization_to_p_weights(polarization)
    TDM_pol = sum(weights[p] * TDM_sel[:, ip] for ip, p in enumerate(p_list))
    TDM_pol_sq = np.abs(TDM_pol)**2

    dE = e.evals_B[:, e_idx, None] - g.evals_B[:, None, g_idx]

    keep_pair = TDM_pol_sq.max(axis=0) >= tdm_thresh   # (n_e_sel, n_g_sel)

    g_label = {gi: _qn_label(g, gi) for gi in g_idx}
    e_label = {ej: _qn_label(e, ej) for ej in e_idx}
    pair_label = {(ej, gi): f"g({g_label[gi]}) → e({e_label[ej]})"
                  for ej in e_idx for gi in g_idx}

    return {
        'Bz': Bz,
        'g_idx': np.asarray(g_idx),
        'e_idx': np.asarray(e_idx),
        'dE': dE,
        'BR': BR,
        'TDM_pol_sq': TDM_pol_sq,
        'leakage': leakage,
        'keep_pair': keep_pair,
        'g_label': g_label,
        'e_label': e_label,
        'pair_label': pair_label,
        'meta': dict(polarization=polarization, q_electronic=tuple(q_electronic),
                     parity_g=parity_g, parity_e=parity_e, tdm_thresh=tdm_thresh),
    }

In [ ]:
_data_test = compute_transitions_vs_B(
    g, e, {'N': [0]}, {'J': 0.5},
    polarization='sigma+', q_electronic=(-1, +1), parity_e='+',
)
N_B = len(Bz)
n_e_sel = len(_data_test['e_idx'])
n_g_sel = len(_data_test['g_idx'])
assert _data_test['dE'].shape         == (N_B, n_e_sel, n_g_sel)
assert _data_test['BR'].shape         == (N_B, n_e_sel, n_g_sel)
assert _data_test['TDM_pol_sq'].shape == (N_B, n_e_sel, n_g_sel)
assert _data_test['leakage'].shape    == (N_B, n_e_sel)
assert (_data_test['BR'] >= 0).all() and (_data_test['BR'] <= 1 + 1e-9).all()
closure = _data_test['BR'].sum(axis=2) + _data_test['leakage']
assert np.allclose(closure, 1.0, atol=1e-6), f"closure max err {np.max(np.abs(closure-1))}"
print("compute_transitions_vs_B: shapes OK, BR closure OK")

In [ ]:
def _grid_shape(n):
    cols = int(np.ceil(np.sqrt(n)))
    rows = int(np.ceil(n / cols))
    return rows, cols


def plot_transition_energies(data, GHz=False, kG=False, figsize_per=(5, 4)):
    """One panel per excited level. Lines = ground partners. y = absolute ΔE."""
    Bz = data['Bz']; e_idx = data['e_idx']; g_idx = data['g_idx']
    dE = data['dE']; keep = data['keep_pair']
    pair_label = data['pair_label']; e_label = data['e_label']

    x = Bz * (1e-3 if kG else 1.0)
    yscale = 1e-3 if GHz else 1.0
    xlabel = 'B (kGauss)' if kG else 'B (Gauss)'
    ylabel = 'ΔE (GHz)' if GHz else 'ΔE (MHz)'

    nrows, ncols = _grid_shape(len(e_idx))
    fig, axes = plt.subplots(nrows, ncols, squeeze=False,
                             figsize=(ncols*figsize_per[0], nrows*figsize_per[1]))
    panels = []
    for k, ej in enumerate(e_idx):
        ax = axes[k // ncols, k % ncols]
        partners = [gi for gi in g_idx if keep[k, list(g_idx).index(gi)]]
        if not partners:
            ax.set_title(f"e({e_label[ej]})  (no allowed pairs)")
            ax.axis('off')
            panels.append({'x': x, 'y': np.empty((len(x), 0)),
                           'e_idx': ej, 'partner_idx': [], 'label': e_label[ej], 'ax': ax})
            continue
        partner_cols = [list(g_idx).index(gi) for gi in partners]
        y = yscale * dE[:, k, partner_cols]
        for col, gi in zip(range(y.shape[1]), partners):
            ax.plot(x, y[:, col], lw=1, label=pair_label[(ej, gi)])
        ax.set_title(f"e({e_label[ej]})", fontsize=10)
        ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
        ax.legend(fontsize=7, loc='best')
        panels.append({'x': x, 'y': y, 'e_idx': ej,
                       'partner_idx': partners, 'label': e_label[ej], 'ax': ax})
    for k in range(len(e_idx), nrows*ncols):
        axes[k // ncols, k % ncols].axis('off')
    fig.suptitle(f"Transition energies — pol={data['meta']['polarization']}, q={data['meta']['q_electronic']}",
                 fontsize=12)
    fig.tight_layout()
    return fig, panels

In [ ]:
_fig, _panels = plot_transition_energies(_data_test)
assert len(_panels) == len(_data_test['e_idx'])
for p in _panels:
    assert p['y'].shape[0] == len(_data_test['Bz'])
plt.close(_fig)
print("plot_transition_energies: OK")

In [ ]:
def plot_branching_ratios(data, kG=False, figsize_per=(5, 4)):
    """One panel per excited level. Lines = BR(e -> g) per selected ground.
    Dashed line per panel = leakage (1 - sum_g BR over selected g)."""
    Bz = data['Bz']; e_idx = data['e_idx']; g_idx = data['g_idx']
    BR = data['BR']; leakage = data['leakage']; keep = data['keep_pair']
    pair_label = data['pair_label']; e_label = data['e_label']

    x = Bz * (1e-3 if kG else 1.0)
    xlabel = 'B (kGauss)' if kG else 'B (Gauss)'

    nrows, ncols = _grid_shape(len(e_idx))
    fig, axes = plt.subplots(nrows, ncols, squeeze=False,
                             figsize=(ncols*figsize_per[0], nrows*figsize_per[1]))
    panels = []
    for k, ej in enumerate(e_idx):
        ax = axes[k // ncols, k % ncols]
        partners = [gi for gi in g_idx if keep[k, list(g_idx).index(gi)]]
        partner_cols = [list(g_idx).index(gi) for gi in partners]
        if partners:
            y = BR[:, k, partner_cols]
            for col, gi in zip(range(y.shape[1]), partners):
                ax.plot(x, y[:, col], lw=1, label=pair_label[(ej, gi)])
        ax.plot(x, leakage[:, k], lw=1, ls='--', color='k', label='leakage')
        ax.set_title(f"e({e_label[ej]})", fontsize=10)
        ax.set_xlabel(xlabel); ax.set_ylabel('Branching ratio')
        ax.set_ylim(0, 1.05)
        ax.legend(fontsize=7, loc='best')
        panels.append({'x': x,
                       'y': BR[:, k, partner_cols] if partners else np.empty((len(x), 0)),
                       'leakage': leakage[:, k],
                       'e_idx': ej, 'partner_idx': partners,
                       'label': e_label[ej], 'ax': ax})
    for k in range(len(e_idx), nrows*ncols):
        axes[k // ncols, k % ncols].axis('off')
    fig.suptitle(f"Spontaneous-emission BR — q={data['meta']['q_electronic']}", fontsize=12)
    fig.tight_layout()
    return fig, panels

In [ ]:
_fig, _panels = plot_branching_ratios(_data_test)
assert len(_panels) == len(_data_test['e_idx'])
for p in _panels:
    assert p['leakage'].shape[0] == len(_data_test['Bz'])
plt.close(_fig)
print("plot_branching_ratios: OK")

In [ ]:
def plot_excitation_strength(data, kG=False, figsize_per=(5, 4)):
    """One panel per ground level. Lines = |TDM_polarization|^2 to excited partners."""
    Bz = data['Bz']; g_idx = data['g_idx']; e_idx = data['e_idx']
    TDM2 = data['TDM_pol_sq']; keep = data['keep_pair']
    pair_label = data['pair_label']; g_label = data['g_label']

    x = Bz * (1e-3 if kG else 1.0)
    xlabel = 'B (kGauss)' if kG else 'B (Gauss)'
    pol = data['meta']['polarization']

    nrows, ncols = _grid_shape(len(g_idx))
    fig, axes = plt.subplots(nrows, ncols, squeeze=False,
                             figsize=(ncols*figsize_per[0], nrows*figsize_per[1]))
    panels = []
    for k, gi in enumerate(g_idx):
        ax = axes[k // ncols, k % ncols]
        partners = [ej for ej in e_idx if keep[list(e_idx).index(ej), k]]
        partner_rows = [list(e_idx).index(ej) for ej in partners]
        if partners:
            y = TDM2[:, partner_rows, k]
            for col, ej in zip(range(y.shape[1]), partners):
                ax.plot(x, y[:, col], lw=1, label=pair_label[(ej, gi)])
        else:
            ax.text(0.5, 0.5, 'no allowed pairs', ha='center', va='center',
                    transform=ax.transAxes)
            y = np.empty((len(x), 0))
        ax.set_title(f"g({g_label[gi]})", fontsize=10)
        ax.set_xlabel(xlabel); ax.set_ylabel(r'$|\mathrm{TDM}_{\mathrm{pol}}|^2$')
        ax.legend(fontsize=7, loc='best')
        panels.append({'x': x, 'y': y, 'g_idx': gi,
                       'partner_idx': partners, 'label': g_label[gi], 'ax': ax})
    for k in range(len(g_idx), nrows*ncols):
        axes[k // ncols, k % ncols].axis('off')
    fig.suptitle(f"Excitation strength — pol={pol}, q={data['meta']['q_electronic']}",
                 fontsize=12)
    fig.tight_layout()
    return fig, panels

In [ ]:
_fig, _panels = plot_excitation_strength(_data_test)
assert len(_panels) == len(_data_test['g_idx'])
plt.close(_fig)
print("plot_excitation_strength: OK")